In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression, LinearRegression, Lasso, Ridge
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV, cross_val_score

In [2]:
# Suppress Warnings for clean notebook
import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv('Melbourne_housing_FULL.csv')

In [4]:
df.head()

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,68 Studley St,2,h,NaN,SS,Jellis,3/09/2016,2.5,3067.0,...,1.0,1.0,126.0,NaN,NaN,Yarra City Council,-37.8014,144.9958,Northern Metropolitan,4019.0
1,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra City Council,-37.7996,144.9984,Northern Metropolitan,4019.0
2,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra City Council,-37.8079,144.9934,Northern Metropolitan,4019.0
3,Abbotsford,18/659 Victoria St,3,u,NaN,VB,Rounds,4/02/2016,2.5,3067.0,...,2.0,1.0,0.0,NaN,NaN,Yarra City Council,-37.8114,145.0116,Northern Metropolitan,4019.0
4,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra City Council,-37.8093,144.9944,Northern Metropolitan,4019.0


In [5]:
df.nunique()

Suburb             351
Address          34009
Rooms               12
Type                 3
Price             2871
Method               9
SellerG            388
Date                78
Distance           215
Postcode           211
Bedroom2            15
Bathroom            11
Car                 15
Landsize          1684
BuildingArea       740
YearBuilt          160
CouncilArea         33
Lattitude        13402
Longtitude       14524
Regionname           8
Propertycount      342
dtype: int64

In [6]:
cols_to_use = ['Suburb', 'Rooms', 'Type', 'Method', 'SellerG', 'Regionname', 'Propertycount', 
               'Distance', 'CouncilArea', 'Bedroom2', 'Bathroom', 'Car', 'Landsize', 'BuildingArea', 'Price']
df = df[cols_to_use]

In [7]:
df.head()

,Suburb,Rooms,Type,Method,SellerG,Regionname,Propertycount,Distance,CouncilArea,Bedroom2,Bathroom,Car,Landsize,BuildingArea,Price
0,Abbotsford,2,h,SS,Jellis,Northern Metropolitan,4019.0,2.5,Yarra City Council,2.0,1.0,1.0,126.0,NaN,NaN
1,Abbotsford,2,h,S,Biggin,Northern Metropolitan,4019.0,2.5,Yarra City Council,2.0,1.0,1.0,202.0,NaN,1480000.0
2,Abbotsford,2,h,S,Biggin,Northern Metropolitan,4019.0,2.5,Yarra City Council,2.0,1.0,0.0,156.0,79.0,1035000.0
3,Abbotsford,3,u,VB,Rounds,Northern Metropolitan,4019.0,2.5,Yarra City Council,3.0,2.0,1.0,0.0,NaN,NaN
4,Abbotsford,3,h,SP,Biggin,Northern Metropolitan,4019.0,2.5,Yarra City Council,3.0,2.0,0.0,134.0,150.0,1465000.0


In [9]:
df = df.rename(columns={'Suburb': 'suburb', 'Rooms': 'rooms', 'Type': 'type', 'Method': 'method', 'SellerG': 'seller_g', 
                        'Regionname': 'region_name', 'Propertycount': 'property_count', 'Distance': 'distance',
                        'CouncilArea': 'council_area', 'Bedroom2': 'bedroom_2', 'Bathroom': 'bathroom', 
                        'Car': 'car', 'Landsize': 'landsize', 'BuildingArea': 'building_area', 'Price': 'price'
                       })

In [10]:
df.isna().sum()

suburb                0
rooms                 0
type                  0
method                0
seller_g              0
region_name           3
property_count        3
distance              1
council_area          3
bedroom_2          8217
bathroom           8226
car                8728
landsize          11810
building_area     21115
price              7610
dtype: int64

In [21]:
df.head()

,suburb,rooms,type,method,seller_g,region_name,property_count,distance,council_area,bedroom_2,bathroom,car,landsize,building_area,price
0,Abbotsford,2,h,SS,Jellis,Northern Metropolitan,4019.0,2.5,Yarra City Council,2.0,1.0,1.0,126.0,160.26,NaN
1,Abbotsford,2,h,S,Biggin,Northern Metropolitan,4019.0,2.5,Yarra City Council,2.0,1.0,1.0,202.0,160.26,1480000.0
2,Abbotsford,2,h,S,Biggin,Northern Metropolitan,4019.0,2.5,Yarra City Council,2.0,1.0,0.0,156.0,79.00,1035000.0
3,Abbotsford,3,u,VB,Rounds,Northern Metropolitan,4019.0,2.5,Yarra City Council,3.0,2.0,1.0,0.0,160.26,NaN
4,Abbotsford,3,h,SP,Biggin,Northern Metropolitan,4019.0,2.5,Yarra City Council,3.0,2.0,0.0,134.0,150.00,1465000.0


In [22]:
df.shape

(34857, 15)

In [11]:
df = df.fillna({'property_count': 0.0, 'distance': 0.0, 'bedroom_2': 0.0, 'bathroom': 0.0, 'car': 0.0})

In [18]:
avg_land = df['landsize'].mean()
avg_building = df['building_area'].mean()
df['landsize'] = round(df['landsize'].fillna(avg_land), 2)
df['building_area'] = round(df['building_area'].fillna(avg_building), 2)

In [23]:
df = df.dropna()

In [24]:
df.isna().sum()

suburb            0
rooms             0
type              0
method            0
seller_g          0
region_name       0
property_count    0
distance          0
council_area      0
bedroom_2         0
bathroom          0
car               0
landsize          0
building_area     0
price             0
dtype: int64

In [25]:
df.shape

(27244, 15)

In [26]:
df = pd.get_dummies(df, drop_first=True, dtype='int')
df.head()

,rooms,property_count,distance,bedroom_2,bathroom,car,landsize,building_area,price,suburb_Aberfeldie,...,council_area_Moorabool Shire Council,council_area_Moreland City Council,council_area_Nillumbik Shire Council,council_area_Port Phillip City Council,council_area_Stonnington City Council,council_area_Whitehorse City Council,council_area_Whittlesea City Council,council_area_Wyndham City Council,council_area_Yarra City Council,council_area_Yarra Ranges Shire Council
1,2,4019.0,2.5,2.0,1.0,1.0,202.0,160.26,1480000.0,0,...,0,0,0,0,0,0,0,0,1,0
2,2,4019.0,2.5,2.0,1.0,0.0,156.0,79.00,1035000.0,0,...,0,0,0,0,0,0,0,0,1,0
4,3,4019.0,2.5,3.0,2.0,0.0,134.0,150.00,1465000.0,0,...,0,0,0,0,0,0,0,0,1,0
5,3,4019.0,2.5,3.0,2.0,1.0,94.0,160.26,850000.0,0,...,0,0,0,0,0,0,0,0,1,0
6,4,4019.0,2.5,3.0,1.0,2.0,120.0,142.00,1600000.0,0,...,0,0,0,0,0,0,0,0,1,0


### Linear Regression

In [38]:
X = df.drop(columns=['price'], axis=1)
y = df['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.3, random_state= 2)
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
print(f'Linear Regression Test_Score: {lin_reg.score(X_test, y_test)}')
print(f'Linear Regression Train_Score: {lin_reg.score(X_train, y_train)}')

Linear Regression Test_Score: 0.13853845669455478
Linear Regression Train_Score: 0.6827792423409915


### Using Lasso Regression Model

In [39]:
lasso_reg = Lasso(alpha=50, max_iter=100, tol=0.1)
lasso_reg.fit(X_train, y_train)
print(f'Lasso Regression Test_Score: {lasso_reg.score(X_test, y_test)}')
print(f'Lasso Regression Train_Score: {lasso_reg.score(X_train, y_train)}')

Lasso Regression Test_Score: 0.6636108051191776
Lasso Regression Train_Score: 0.6766985771854733


### Using Ridge Regression Model

In [40]:
ridge_reg= Ridge(alpha=50, max_iter=100, tol=0.1)
ridge_reg.fit(X_train, y_train)
print(f'Ridge Regression Test_Score: {ridge_reg.score(X_test, y_test)}')
print(f'Ridge Regression Train_Score: {ridge_reg.score(X_train, y_train)}')

Ridge Regression Test_Score: 0.6670848977814987
Ridge Regression Train_Score: 0.6622376753851893


##### We see that Lasso and Ridge Regularizations prove to be beneficial when our Simple Linear Regression Model overfits. These results may not be that contrast but significant in most cases.Also that L1 & L2 Regularizations are used in Neural Networks too